In [ ]:
import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# 1. Define JSON structure schema
edu_schema = pl.List(
    pl.Struct([
        pl.Field("degree_user_input", pl.String),
        pl.Field("degree_standardized", pl.String),
        pl.Field("university", pl.String),
        pl.Field("major_user_input", pl.String),
        pl.Field("major_standardized", pl.String),
        pl.Field("description", pl.String),
        pl.Field("start_date", pl.String),
        pl.Field("end_date", pl.String)
    ])
)

# 2. Parse JSON array and flatten entries
df_clean = (
    pl.read_csv("./fresh-grad-filter.csv")
    .with_columns(pl.col("educations").str.json_decode(edu_schema))
    .explode("educations")
    .unnest("educations")
)

# 3. Calculate how many missing majors each user has
user_missing_counts = (
    df_clean
    .group_by("id")
    .agg(
        pl.col("major_standardized").is_null().sum().alias("missing_edu_count")
    )
)

# 4. Separate users into "Complete" or categorize by their missing entry volumes
complete_count = user_missing_counts.filter(pl.col("missing_edu_count") == 0).height

# Pull exact user IDs for the missing categories
missing_df = user_missing_counts.filter(pl.col("missing_edu_count") > 0).sort("missing_edu_count")

# Create buckets for statistics and charts using a bulletproof rule-based sorting index
missing_summary = (
    user_missing_counts
    .with_columns(
        pl.when(pl.col("missing_edu_count") == 0).then(pl.lit("Complete"))
        .when(pl.col("missing_edu_count") == 1).then(pl.lit("1 Missing"))
        .when(pl.col("missing_edu_count") == 2).then(pl.lit("2 Missing"))
        .otherwise(pl.lit("3+ Missing"))
        .alias("Status"),
        
        pl.when(pl.col("missing_edu_count") == 0).then(0)
        .when(pl.col("missing_edu_count") == 1).then(1)
        .when(pl.col("missing_edu_count") == 2).then(2)
        .otherwise(3)
        .alias("sort_idx")
    )
    .group_by(["Status", "sort_idx"])
    .agg(pl.len().alias("User Count"))
    .sort("sort_idx")
)

# 5. PRINT OUTPUT TERMINAL METRICS
print("─── SUMMARY METRICS ───")
print(f"Total Users with Complete Educations: {complete_count}\n")

print("─── USERS WITH MISSING EDUCATION DATA ───")
unique_missing_values = sorted(missing_df["missing_edu_count"].unique().to_list())

for count_val in unique_missing_values:
    ids_list = missing_df.filter(pl.col("missing_edu_count") == count_val)["id"].to_list()
    suffix = "entry" if count_val == 1 else "entries"
    print(f"\n[Users missing exactly {count_val} {suffix} ({len(ids_list)} total)]")
    print(", ".join(ids_list))

# 6. GENERATE AND SAVE PLOT CHART
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(
    data=missing_summary.to_pandas(),
    x="Status",
    y="User Count",
    hue="Status",
    palette="coolwarm",
    ax=ax,
    legend=False
)

ax.set_title("Data Completeness: Missing 'major_standardized' Entries Per User Profile", fontsize=12, fontweight="bold")
ax.set_xlabel("Profile Completeness Category")
ax.set_ylabel("Number of Users")

# Label bar heights explicitly
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{int(p.get_height())}", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha="center", va="center", xytext=(0, 8), 
                    textcoords="offset points", fontweight="bold")

plt.tight_layout()
plt.savefig("education_completeness_chart.png", dpi=300)
plt.close()
print("\n[Chart successfully generated and exported to 'education_completeness_chart.png']")

In [ ]:
import polars as pl

# 1. Load the dataset
df = pl.read_csv("../Data/final.csv")


# 3. Count occurrences for an array of university names
target_universities = ["BUV", "British University Vietnam", "Swinburne", "FUV", "Fulbright University Vietnam", "RMIT","Royal Melbourne Institute of Technology Vietnam","VinUniversity","VIN"]

# Parse the JSON strings, explode the list into individual rows, and extract the university field
df_unis = (
    df.select(pl.col("educations").str.json_decode(edu_schema))
    .explode("educations")
    .select(pl.col("educations").struct.field("university").str.to_lowercase().alias("uni_name"))
    .filter(pl.col("uni_name").is_not_null())
)

# Compute the case-insensitive substring match count for each name in the array
university_counts = {
    uni: df_unis.filter(pl.col("uni_name").str.contains(uni.lower())).height
    for uni in target_universities
}

print("University Occurrences:", university_counts)

In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Load and parse data from output.csv
df = pd.read_csv("output.csv")
data_list = []

for col in df.columns:
    if col == "Rank":
        continue
    for val in df[col].dropna():
        # Regular expression to extract the institution name and user count
        match = re.search(r"^(.*) \((\d+) users\)$", str(val))
        if match:
            inst = match.group(1).strip()
            count = int(match.group(2))
            data_list.append(
                {"Degree": col, "Institution": inst, "User Count": count}
            )

parsed_df = pd.DataFrame(data_list)
sns.set_theme(style="whitegrid")

# Chart 1: Overall Distribution of Degree Tiers
degree_summary = (
    parsed_df.groupby("Degree")["User Count"]
    .sum()
    .reset_index()
    .sort_values(by="User Count", ascending=False)
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=degree_summary,
    x="User Count",
    y="Degree",
    hue="Degree",
    palette="viridis",
    ax=ax,
)
ax.set_title(
    "Total User Distribution Across Degree Tiers", fontsize=14, fontweight="bold"
)
ax.set_xlabel("Total User Count")
ax.set_ylabel("Degree Tier")
if ax.get_legend() is not None:
    ax.get_legend().remove()
for i, v in enumerate(degree_summary["User Count"]):
    ax.text(v + 5, i, f" {v}", va="center", fontweight="bold")
plt.tight_layout()
plt.savefig("degree_distribution.png", dpi=300)
plt.close()

# Chart 2: Top Bachelor's Degree Institutions
top_bachelor = (
    parsed_df[parsed_df["Degree"] == "bachelor"]
    .sort_values(by="User Count", ascending=False)
    .head(15)
)
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    data=top_bachelor,
    x="User Count",
    y="Institution",
    hue="Institution",
    palette="Blues_r",
    ax=ax,
)
ax.set_title(
    "Top 15 Most Popular Institutions for Bachelor's Degrees",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("User Count")
ax.set_ylabel("Institution")
if ax.get_legend() is not None:
    ax.get_legend().remove()
for i, v in enumerate(top_bachelor["User Count"]):
    ax.text(v + 5, i, f" {v}", va="center")
plt.tight_layout()
plt.savefig("bachelor_top_institutions.png", dpi=300)
plt.close()

# Chart 3: Grid Plot for Other Key Educational Tracks
fig, axes = plt.subplots(3, 1, figsize=(12, 16))
tiers = ["college", "high school", "master"]
titles = [
    "Top 10 Most Popular College Track Institutions",
    "Top 10 Most Popular High School Institutions",
    "Top 10 Most Popular Master's Degree Institutions",
]
palettes = ["Oranges_r", "Greens_r", "Purples_r"]

for idx, tier in enumerate(tiers):
    tier_data = (
        parsed_df[parsed_df["Degree"] == tier]
        .sort_values(by="User Count", ascending=False)
        .head(10)
    )
    sns.barplot(
        data=tier_data,
        x="User Count",
        y="Institution",
        hue="Institution",
        palette=palettes[idx],
        ax=axes[idx],
    )
    axes[idx].set_title(titles[idx], fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("User Count")
    axes[idx].set_ylabel("Institution")
    if axes[idx].get_legend() is not None:
        axes[idx].get_legend().remove()
    for i, v in enumerate(tier_data["User Count"]):
        axes[idx].text(v + 0.2, i, f" {v}", va="center")

plt.tight_layout()
plt.savefig("other_degrees_top_institutions.png", dpi=300)
plt.close()

In [ ]:
import polars as pl

# 1. Define schema
edu_schema = pl.List(
    pl.Struct([
        pl.Field("degree_user_input", pl.String),
        pl.Field("degree_standardized", pl.String),
        pl.Field("university", pl.String),
        pl.Field("major_user_input", pl.String),
        pl.Field("major_standardized", pl.String),
        pl.Field("description", pl.String),
        pl.Field("start_date", pl.String),
        pl.Field("end_date", pl.String)
    ])
)

# 2. Process and Clean 
df_clean = (
    pl.read_csv("./fresh-grad-filter.csv")
    .with_columns(pl.col("educations").str.json_decode(edu_schema))
    .explode("educations")
    .unnest("educations")
    .with_columns(pl.col("degree_standardized").str.strip_chars().str.to_lowercase())
    .filter(pl.col("degree_standardized").is_not_null() & pl.col("university").is_not_null())
)

# 3. Calculate Popularity Rankings
ranked_data = (
    df_clean
    .unique(subset=["id", "degree_standardized", "university"])
    .group_by(["degree_standardized", "university"])
    .agg(pl.len().alias("count"))
    .sort(["degree_standardized", "count"], descending=[False, True])
)

# 4. Format entries and Pivot into explicit Side-by-Side Table Columns
final_table = (
    ranked_data
    .with_columns(
        pl.format("{} ({} users)", pl.col("university"), pl.col("count")).alias("entry")
    )
    .with_columns(
        pl.int_range(1, pl.len() + 1).over("degree_standardized").alias("Rank")
    )
    .pivot(
        on="degree_standardized",
        index="Rank",
        values="entry",
        aggregate_function="first"
    )
    .sort("Rank")
)

# 5. Export and print cleanly as a true CSV string
final_table.write_csv("output.csv")
print("Successfully generated output.csv")